# RSNA Knee: сборка кэша на Kaggle

Ноутбук делает одну вещь: читает DICOM соревнования и сохраняет кэш картинок для обучения,
около 15 ГБ вместо 570 ГБ исходных данных. GPU ему не нужен, поэтому он не тратит недельную квоту.

Функции подготовки картинок скопированы из `02_report_labels_mil.ipynb` без изменений. Настройки
`IMG`, `CROP_MM`, `K_WIN`, `WIN_BAND` должны совпадать с ноутбуком обучения: при загрузке кэша это проверяется.

## Как запустить

1. Kaggle -> New Notebook -> File -> Import Notebook -> этот файл.
2. Input: соревнование `rsna-knee-abnormality-detection`.
3. Settings: Accelerator **None** (CPU), Internet можно выключить.
4. **Save Version -> Save & Run All (Commit)**. Займёт около 1-1.5 ч.
5. Когда версия досчитается: вкладка **Output -> New Dataset**, имя например `rsna-knee-cache-224`.

## Как использовать кэш

- **На Kaggle GPU:** добавить этот датасет в Input ноутбука `02_report_labels_mil.ipynb`. Он найдёт кэш сам.
- **На Mac:** скачать датасет в папку `data/` рядом с `02_report_labels_mil.ipynb`:

```bash
pip install kaggle
kaggle datasets download -d <ваш-ник>/rsna-knee-cache-224 -p data --unzip
```

Для `kaggle` нужен API-токен: Kaggle -> Settings -> API -> Create New Token, файл `kaggle.json`
положить в `~/.kaggle/` (`chmod 600 ~/.kaggle/kaggle.json`).

## Что окажется в датасете

| Файл | Что это |
|---|---|
| `cache_train.npy` | картинки `[4407, 6, 12, 224, 224]` uint8: исследование, слот, срез, высота, ширина |
| `mask_train.npy` | `[4407, 6]`: 1, если слот заполнен |
| `pos_train.npy` | `[4407, 6, 4]`: где в стеке стоит каждое окно (0-1) |
| `studies_train.json` | порядок исследований в кэше |
| `rsna_cache_meta.json` | настройки, с которыми собран кэш |
| `train.csv`, `train_series.csv`, `test.csv`, `test_series.csv` | таблицы соревнования: локально ноутбуку обучения нужны метки и тексты заключений |

In [ ]:
import os 
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "4")

import gc 
import re
import math 
import time 
import glob 
import json
import hashlib 
import textwrap 
import warnings
from copy import deepcopy 
from pathlib import Path 
from concurrent.futures import ThreadPoolExecutor 

import cv2 
import numpy as np 
import pandas as pd 
import pydicom 
import matplotlib.pyplot as plt 
import torch 
import torch.nn as nn
import torch.nn.functional as F 
import timm 
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
T0 = time.time()

def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)

try:
    display 
except NameError:
    display = print 

# Устройство: видеокарта NVIDIA (Kaggle) -> видеокарта Apple Silicon (MPS) -> процессор
DEV = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")


def empty_device_cache():
    # освободить кэш памяти видеокарты; у CUDA и MPS это разные функции
    if DEV.type == "cuda":
        torch.cuda.empty_cache()
    elif DEV.type == "mps":
        torch.mps.empty_cache()
log(f"device: {DEV}, torch {torch.__version__}, timm {timm.__version__}, pydicom {pydicom.__version__}")

## Настройки

Должны совпадать с конфигом `02_report_labels_mil.ipynb`.

In [ ]:
DATA_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
if not DATA_DIR.exists():
    DATA_DIR = Path("/kaggle/input/rsna-knee-abnormality-detection")
OUT_DIR = Path("/kaggle/working")       # всё, что здесь, станет содержимым датасета

IMG = 224                  # сторона картинки на входе сети (кратна 14 - размер патча ViT)
CROP_MM = 140.0            # вырезаем квадрат 140×140 МИЛЛИМЕТРОВ вокруг центра -> у всех
K_WIN = 4                  # сколько «окон» по 3 соседних среза берём из каждой серии
WIN_BAND = (0.08, 0.92)    # где по стеку ставить окна: от 8% до 92% (края - мягкие ткани,
FILL_MISSING_SLOTS = True  # нет серии с fat-sat -> подставить такую же плоскость без fat-sat (и наоборот)
SERIES_THREADS = 24        # сколько серий читаем параллельно (упор в диск Kaggle, не в CPU)
CACHE_MAX_SHARE = 0.55     # кэш может занять не больше этой доли свободной памяти

log(f"DATA_DIR = {DATA_DIR} (существует: {DATA_DIR.exists()})")

## Функции из `02_report_labels_mil.ipynb`: слоты, чтение серии, кэш

In [ ]:
SLOTS = [  # (имя, плоскость, Fat_Suppression)
    ("SAG_FS", "Sagittal", 1), ("COR_FS", "Coronal", 1), ("AX_FS", "Axial", 1),
    ("SAG_NOFS", "Sagittal", 0), ("COR_NOFS", "Coronal", 0), ("AX_NOFS", "Axial", 0),
]
N_SLOT = len(SLOTS)
SAG_SLOTS = [k for k, (_, plane, _) in enumerate(SLOTS) if plane == "Sagittal"]


def count_files(d):
    try:
        return sum(1 for e in os.scandir(d) if e.name.endswith(".dcm"))
    except OSError:
        return 0


def build_slot_table(series_df, split):
    # -> таблица «исследование, слот -> какая серия», одна строка на заполненный слот
    df = series_df.copy()
    df["dir"] = [str(DATA_DIR / split / str(a) / str(b))
                 for a, b in zip(df["StudyInstanceUID"], df["SeriesInstanceUID"])]
    with ThreadPoolExecutor(SERIES_THREADS) as pool:
        df["n_files"] = list(pool.map(count_files, df["dir"]))
    rows = []
    for study, g in df.groupby("StudyInstanceUID"):
        chosen = {}
        for k, (name, plane, fs) in enumerate(SLOTS):
            cand = g[(g["Anatomical_Plane"] == plane) & (g["Fat_Suppression"].astype(int) == fs)
                     & (g["n_files"] >= 3)]
            if len(cand):
                chosen[k] = (cand.sort_values("n_files", ascending=False).iloc[0], False)
        if FILL_MISSING_SLOTS:
            for a, b in ((0, 3), (1, 4), (2, 5)):          # пары «та же плоскость, FS / без FS»
                if a not in chosen and b in chosen:
                    chosen[a] = (chosen[b][0], True)
                elif b not in chosen and a in chosen:
                    chosen[b] = (chosen[a][0], True)
        for k, (r, filled) in chosen.items():
            rows.append({"StudyInstanceUID": study, "slot": k, "slot_name": SLOTS[k][0],
                         "SeriesInstanceUID": r["SeriesInstanceUID"], "dir": r["dir"],
                         "plane": r["Anatomical_Plane"], "n_files": r["n_files"], "filled": filled})
    return pd.DataFrame(rows)

In [ ]:
HDR_TAGS = ["ImagePositionPatient", "ImageOrientationPatient", "PixelSpacing", "InstanceNumber"]
PLANES = ["Sagittal", "Coronal", "Axial"]
# Каноническая ориентация (из ноутбука 0.958). Для плоскости, у которой нормаль идёт вдоль оси
# 0 (x, лево-право) / 1 (y, перёд-зад) / 2 (z, верх-низ), задаём:
# (знак порядка срезов, (ось вдоль столбцов картинки, знак), (ось вдоль строк, знак))
# Система координат DICOM: x -> к ЛЕВОМУ боку пациента, y -> к спине, z -> к голове.
CANON = {
    0: (+1, (1, +1), (2, -1)),   # сагиттальная: слева направо по картинке = спереди назад, вниз = от головы
    1: (+1, (0, +1), (2, -1)),   # корональная: по картинке справа пациента налево, вниз = от головы
    2: (-1, (0, +1), (1, +1)),   # аксиальная: срезы сверху вниз, по картинке вниз = спереди назад
}


def _vec(v, n):
    try:
        a = np.asarray([float(x) for x in v], float)
        return a if len(a) >= n else None
    except Exception:
        return None


def read_headers(d):
    recs = []
    for f in sorted(os.listdir(d)):
        if not f.endswith(".dcm"):
            continue
        rec = {"file": f, "ipp": None, "iop": None, "ps": None, "inst": None}
        try:
            h = pydicom.dcmread(os.path.join(d, f), stop_before_pixels=True, force=True,
                                specific_tags=HDR_TAGS)
            rec["ipp"] = _vec(getattr(h, "ImagePositionPatient", []), 3)
            rec["iop"] = _vec(getattr(h, "ImageOrientationPatient", []), 6)
            rec["ps"] = _vec(getattr(h, "PixelSpacing", []), 2)
            rec["inst"] = float(getattr(h, "InstanceNumber", None) or 0)
        except Exception:
            pass
        recs.append(rec)
    return recs


def sort_slices(recs, plane_name):
    # -> (отсортированные записи, номер оси плоскости 0/1/2, IOP для ориентации)
    iop = next((r["iop"] for r in recs if r["iop"] is not None), None)
    if iop is not None:
        normal = np.cross(iop[:3], iop[3:])
        ax = int(np.abs(normal).argmax())
        if all(r["ipp"] is not None for r in recs):
            sgn = CANON[ax][0] * np.sign(normal[ax])
            recs = sorted(recs, key=lambda r: float(np.dot(r["ipp"], normal)) * sgn)
            return recs, ax, iop
    else:
        ax = PLANES.index(plane_name) if plane_name in PLANES else 0
    return sorted(recs, key=lambda r: (r["inst"] or 0, r["file"])), ax, iop


def canon_orient(img, iop, ps, ax):
    # поворачиваем/отражаем срез к канонической ориентации; ps (размер пикселя) меняем вместе с ним
    if iop is None:
        return img, ps
    r, c = np.array(iop[:3]), np.array(iop[3:])
    _, (cax, csgn), (rax, rsgn) = CANON[ax]
    if abs(r[cax]) < abs(r[rax]):           # строки и столбцы перепутаны -> транспонируем
        img, r, c, ps = img.T, c, r, ps[::-1]
    if r[cax] * csgn < 0:
        img = img[:, ::-1]                  # отражаем по горизонтали
    if c[rax] * rsgn < 0:
        img = img[::-1]                     # отражаем по вертикали
    return np.ascontiguousarray(img), ps


def crop_resize(img, ps):
    # квадрат CROP_MM мм вокруг центра -> IMG×IMG; если снимок меньше, дополняем нулями
    out = np.zeros((IMG, IMG), np.float32)
    mm_per_px = CROP_MM / IMG
    h, w = img.shape
    ch = min(h, int(round(CROP_MM / ps[0])))
    cw = min(w, int(round(CROP_MM / ps[1])))
    y0, x0 = (h - ch) // 2, (w - cw) // 2
    crop = np.ascontiguousarray(img[y0:y0 + ch, x0:x0 + cw])
    nh = max(1, min(IMG, int(round(ch * ps[0] / mm_per_px))))
    nw = max(1, min(IMG, int(round(cw * ps[1] / mm_per_px))))
    interp = cv2.INTER_AREA if nh < ch else cv2.INTER_LINEAR   # AREA - без «лесенки» при уменьшении
    res = cv2.resize(crop, (nw, nh), interpolation=interp)
    oy, ox = (IMG - nh) // 2, (IMG - nw) // 2
    out[oy:oy + nh, ox:ox + nw] = res
    return out


def window_centres(n):
    # K_WIN центров окон, равномерно в полосе WIN_BAND, так, чтобы c-1 и c+1 существовали
    a = max(1.0, WIN_BAND[0] * (n - 1))
    b = min(n - 2.0, WIN_BAND[1] * (n - 1))
    if b < a:
        a = b = (n - 1) / 2
    return np.clip(np.round(np.linspace(a, b, K_WIN)).astype(int), 0, max(0, n - 1))


def decode(path):
    ds = pydicom.dcmread(path, force=True)
    a = ds.pixel_array.astype(np.float32)
    if a.ndim == 3:                         # редкий «мультикадровый» файл - берём средний кадр
        a = a[len(a) // 2]
    a = a * float(getattr(ds, "RescaleSlope", 1) or 1) + float(getattr(ds, "RescaleIntercept", 0) or 0)
    if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":
        a = a.max() - a                     # в MONOCHROME1 «белое» и «чёрное» перевёрнуты
    return a


DECODE_FAILED = []


def read_series(d, plane_name):
    # -> (uint8 [K_WIN*3, IMG, IMG], позиции окон [K_WIN] от 0 до 1) или None
    recs = read_headers(d)
    if not recs:
        return None
    recs, ax, iop = sort_slices(recs, plane_name)
    n = len(recs)
    centres = window_centres(n)
    idx = np.clip(np.stack([centres - 1, centres, centres + 1], 1).ravel(), 0, n - 1)

    imgs = {}
    for i in sorted(set(idx.tolist())):
        try:
            imgs[i] = decode(os.path.join(d, recs[i]["file"]))
        except Exception:
            imgs[i] = None
    ok = [i for i, a in imgs.items() if a is not None]
    if not ok:
        DECODE_FAILED.append(d)
        return None
    if len(ok) < len(imgs):
        DECODE_FAILED.append(d)
    for i in imgs:                          # сломанный срез -> ближайший целый
        if imgs[i] is None:
            imgs[i] = imgs[min(ok, key=lambda j: abs(j - i))]

    ps_default = next((r["ps"] for r in recs if r["ps"] is not None), None)
    planes = []
    for i in idx:
        a = imgs[int(i)]
        ps = recs[int(i)]["ps"] if recs[int(i)]["ps"] is not None else ps_default
        if ps is None:                      # нет размера пикселя -> считаем, что снимок = кропу
            ps = np.array([CROP_MM / max(a.shape)] * 2)
        a, ps = canon_orient(a, iop, list(ps), ax)
        planes.append((a, ps))

    allpx = np.concatenate([a[::4, ::4].ravel() for a, _ in planes])
    lo, hi = np.percentile(allpx, [1, 99])
    vol = np.stack([crop_resize(np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1), ps) for a, ps in planes])
    pos = centres / max(n - 1, 1)
    return (vol * 255).round().astype(np.uint8), pos.astype(np.float32)

In [ ]:
def available_gb():
    try:
        with open("/proc/meminfo") as fh:
            info = dict(l.split(":", 1) for l in fh if ":" in l)
        return int(info["MemAvailable"].split()[0]) / 1024 ** 2
    except Exception:
        return 16.0


CACHE_KEYS = ("IMG", "CROP_MM", "K_WIN", "WIN_BAND", "FILL_MISSING_SLOTS")


def cache_config():
    # всё, от чего зависят пиксели кэша; должно совпадать при записи и при загрузке
    return {"IMG": IMG, "CROP_MM": CROP_MM, "K_WIN": K_WIN, "WIN_BAND": list(WIN_BAND),
            "FILL_MISSING_SLOTS": FILL_MISSING_SLOTS, "SLOTS": [list(s) for s in SLOTS]}


def build_cache(slot_table, tag, out_dir=None):
    # out_dir=None: кэш в оперативной памяти; out_dir=папка: кэш сразу пишется в файл на диске
    # (так делает 03_build_cache_kaggle.ipynb, чтобы потом сохранить его как датасет)
    global K_WIN
    studies = sorted(slot_table["StudyInstanceUID"].unique())
    need = len(studies) * N_SLOT * K_WIN * 3 * IMG * IMG / 1024 ** 3
    avail = available_gb()
    log(f"{tag}: кэш займёт {need:.1f} ГБ, свободно {avail:.1f} ГБ")
    while out_dir is None and need > CACHE_MAX_SHARE * avail and K_WIN > 2:
        K_WIN -= 1                          # не влезаем - уменьшаем число окон, а не разрешение
        need = len(studies) * N_SLOT * K_WIN * 3 * IMG * IMG / 1024 ** 3
        log(f"   не влезает: K_WIN уменьшен до {K_WIN} ({need:.1f} ГБ)")

    sidx = {s: i for i, s in enumerate(studies)}
    shape = (len(studies), N_SLOT, K_WIN * 3, IMG, IMG)
    if out_dir is None:
        cache = np.zeros(shape, np.uint8)
    else:
        out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
        cache = np.lib.format.open_memmap(out_dir / f"cache_{tag}.npy", mode="w+", dtype=np.uint8, shape=shape)
    mask = np.zeros((len(studies), N_SLOT), np.float32)
    pos = np.zeros((len(studies), N_SLOT, K_WIN), np.float32)

    # одна серия может стоять в двух слотах (подстановка) - читаем её один раз
    uniq = slot_table.drop_duplicates("SeriesInstanceUID")[["SeriesInstanceUID", "dir", "plane"]]
    where = slot_table.groupby("SeriesInstanceUID")[["StudyInstanceUID", "slot"]].apply(
        lambda g: list(zip(g["StudyInstanceUID"], g["slot"]))).to_dict()
    jobs = list(uniq.itertuples(index=False))
    log(f"{tag}: читаю {len(jobs)} серий в {SERIES_THREADS} потоков …")
    t0, done = time.time(), 0
    with ThreadPoolExecutor(SERIES_THREADS) as pool:
        for c0 in range(0, len(jobs), 256):
            block = jobs[c0:c0 + 256]
            for j, out in zip(block, pool.map(lambda j: read_series(j.dir, j.plane), block)):
                done += 1
                if out is None:
                    continue
                vol, p = out
                for st, k in where[j.SeriesInstanceUID]:
                    cache[sidx[st], k], pos[sidx[st], k], mask[sidx[st], k] = vol, p, 1.0
            if (c0 // 256) % 10 == 0 or done == len(jobs):
                rate = done / max(time.time() - t0, 1e-6)
                log(f"  {tag}: {done}/{len(jobs)} серий, {rate:.1f} серий/с, "
                    f"осталось ≈{(len(jobs) - done) / max(rate, 1e-6) / 60:.0f} мин")
    log(f"{tag}: заполнено слотов {int(mask.sum())}, серий с битыми срезами: {len(DECODE_FAILED)}")
    gc.collect()
    if out_dir is not None:
        cache.flush()
        np.save(out_dir / f"mask_{tag}.npy", mask)
        np.save(out_dir / f"pos_{tag}.npy", pos)
        with open(out_dir / f"studies_{tag}.json", "w") as fh:
            json.dump(studies, fh)
        with open(out_dir / "rsna_cache_meta.json", "w") as fh:
            json.dump(cache_config(), fh, indent=1)
        log(f"{tag}: кэш записан в {out_dir}")
    return studies, cache, mask, pos


def load_cache(cache_dir, tag):
    # готовый кэш из датасета: проверяем, что он собран с теми же настройками, что в конфиге
    cache_dir = Path(cache_dir)
    with open(cache_dir / "rsna_cache_meta.json") as fh:
        meta = json.load(fh)
    now = cache_config()
    diff = {k: (meta.get(k), now[k]) for k in now if meta.get(k) != now[k]}
    assert not diff, f"кэш собран с другими настройками (в кэше, в конфиге): {diff}"
    with open(cache_dir / f"studies_{tag}.json") as fh:
        studies = json.load(fh)
    cache = np.load(cache_dir / f"cache_{tag}.npy", mmap_mode=None if CACHE_IN_RAM else "r")
    mask = np.load(cache_dir / f"mask_{tag}.npy")
    pos = np.load(cache_dir / f"pos_{tag}.npy")
    log(f"{tag}: кэш загружен из {cache_dir}: {cache.shape}, "
        f"{'в памяти' if CACHE_IN_RAM else 'с диска через memmap'}")
    return studies, cache, mask, pos

## Сборка

Сначала таблица слотов, потом чтение серий прямо в файл `cache_train.npy`. Прогресс и оставшееся время печатаются в логе.

In [ ]:
import shutil

train_series = pd.read_csv(DATA_DIR / "train_series.csv")
free_gb = shutil.disk_usage(OUT_DIR).free / 1024 ** 3
need_gb = train_series["StudyInstanceUID"].nunique() * N_SLOT * K_WIN * 3 * IMG * IMG / 1024 ** 3
log(f"нужно ~{need_gb:.1f} ГБ, свободно в {OUT_DIR}: {free_gb:.1f} ГБ")
assert need_gb < free_gb, "кэш не помещается: уменьшите IMG или K_WIN"

SLOT_TR = build_slot_table(train_series, "train_series")
ST_TR, CTR, MTR, PTR = build_cache(SLOT_TR, "train", out_dir=OUT_DIR)

for name in ("train.csv", "train_series.csv", "test.csv", "test_series.csv"):
    shutil.copy(DATA_DIR / name, OUT_DIR / name)
for p in sorted(OUT_DIR.iterdir()):
    print(f"  {p.name:<24} {p.stat().st_size / 1024 ** 2:>10.1f} МБ")

## Проверка глазами

Центральный срез каждого окна для одного исследования, прямо из записанного кэша.

In [ ]:
check = np.load(OUT_DIR / "cache_train.npy", mmap_mode="r")
i = len(ST_TR) // 2
fig, axes = plt.subplots(N_SLOT, K_WIN, figsize=(2.4 * K_WIN, 2.4 * N_SLOT))
for s in range(N_SLOT):
    for j in range(K_WIN):
        axes[s, j].imshow(check[i, s, j * 3 + 1], cmap="gray")
        axes[s, j].set_title(f"{SLOTS[s][0]} окно {j} mask={MTR[i, s]:.0f}", fontsize=7)
        axes[s, j].axis("off")
plt.tight_layout(); plt.show()